In [ ]:
import sys
print(sys.executable)

import sklearn
print(sklearn.__version__)

In [ ]:
# ============================================================
# RUNTIME BENCHMARK LOGGER
# ============================================================

from time import perf_counter
from datetime import datetime
from pathlib import Path
import pandas as pd
import os

RUNTIME_FILE = Path("FINAL_experimental_runtime_summary.csv")

if RUNTIME_FILE.exists():
    RUNTIME_LOG = pd.read_csv(RUNTIME_FILE).to_dict("records")
else:
    RUNTIME_LOG = []


def start_timer(experiment_name):
    print("\n" + "=" * 75)
    print(f"STARTING: {experiment_name}")
    print("Clock time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    print("=" * 75)

    return perf_counter()


def stop_timer(experiment_name, start_time):
    elapsed_sec = perf_counter() - start_time

    record = {
        "experiment": experiment_name,
        "runtime_seconds": elapsed_sec,
        "runtime_minutes": elapsed_sec / 60.0,
        "runtime_hours": elapsed_sec / 3600.0,
    }

    RUNTIME_LOG.append(record)

    pd.DataFrame(RUNTIME_LOG).to_csv(
        RUNTIME_FILE,
        index=False
    )

    print("\n" + "-" * 75)
    print(f"FINISHED: {experiment_name}")
    print(f"Runtime: {elapsed_sec:.2f} seconds")
    print(f"Runtime: {elapsed_sec / 60.0:.4f} minutes")
    print("-" * 75)

    return elapsed_sec

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, confusion_matrix
)

In [ ]:
import sys
print(sys.executable)

In [ ]:
df = pd.read_csv("../Data/Raw/triage_cleaned_working.csv")

In [ ]:
print(df.shape)
df.head()

Define feature groups

In [ ]:
target_col = "y"

numeric_cols = [
    "age", "n_admissions",
    "heart_rate_final", "bp_systolic", "bp_diastole",
    "respiratory", "temperature", "spo2",
    "vitals_missing_count"
]

categorical_cols = [
    "arrival_mode", "arrival_month", "arrival_day", "arrival_hour_bin"
]

binary_cols = [
    "abdominal_pain", "chest_pain", "back_pain", "unclassified",
    "ckd", "dm", "headache", "htn", "cough", "chd", "asthma",
    "bronchitis", "dizziness", "glaucoma", "hepatitis", "paralysis",
    "pneumonia", "sicklecell", "arm_pain", "breast_pain", "dental_pain",
    "diarrhea", "ear_pain", "eye_pain", "fever", "pelvic_pain",
    
    # missingness / data-quality
    "arrival_mode_missing",
    "hr_missing", "pulse_rate_missing", "bp_systolic_missing",
    "bp_diastole_missing", "respiratory_missing", "temperature_missing", "spo2_missing",
    "heart_rate_disagreement_flag"
]

feature_cols = numeric_cols + categorical_cols + binary_cols

print("Total features before encoding:", len(feature_cols))

Final dataset for modeling

In [ ]:
model_df = df[feature_cols + [target_col]].copy()

print(model_df.shape)
print(model_df[target_col].value_counts(normalize=True))

Train / validation / test split

In [ ]:
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df[target_col],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[target_col],
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(name, d[target_col].mean())

Separate X and y

In [ ]:
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

Build preprocessing pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
        ("bin", binary_transformer, binary_cols),
    ]
)

Define baseline model

In [ ]:
logreg = LogisticRegression(
    max_iter=3000,
    solver="lbfgs",
    class_weight=None,
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", logreg)
])

Fit baseline and evaluate on validation

In [ ]:
pipeline.fit(X_train, y_train)

val_pred_proba = pipeline.predict_proba(X_val)[:, 1]
test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

def evaluate_binary(y_true, p, threshold=0.5):
    y_pred = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    
    return {
        "AUROC": roc_auc_score(y_true, p),
        "AUPRC": average_precision_score(y_true, p),
        "Brier": brier_score_loss(y_true, p),
        "Recall@0.5": recall,
        "Precision@0.5": precision
    }

print("Validation:", evaluate_binary(y_val, val_pred_proba))
print("Test:", evaluate_binary(y_test, test_pred_proba))

Generate OOF predictions on training set

In [ ]:
t_runtime_oof = start_timer(
    "Baseline five-fold OOF modelling"
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_pred = np.zeros(len(X_train))

for fold, (tr_idx, oof_idx) in enumerate(skf.split(X_train, y_train), start=1):
    X_tr, X_oof = X_train.iloc[tr_idx], X_train.iloc[oof_idx]
    y_tr, y_oof = y_train.iloc[tr_idx], y_train.iloc[oof_idx]
    
    fold_pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=3000, solver="lbfgs", random_state=42))
    ])
    
    fold_pipeline.fit(X_tr, y_tr)
    oof_pred[oof_idx] = fold_pipeline.predict_proba(X_oof)[:, 1]
    
    print(f"Fold {fold} done")
stop_timer(
    "Baseline five-fold OOF modelling",
    t_runtime_oof
)

training diagnostic table

In [ ]:
diag_train = train_df.copy().reset_index(drop=True)

diag_train["oof_pred_proba"] = oof_pred
diag_train["oof_logloss"] = -(
    diag_train["y"] * np.log(np.clip(diag_train["oof_pred_proba"], 1e-8, 1-1e-8)) +
    (1 - diag_train["y"]) * np.log(np.clip(1 - diag_train["oof_pred_proba"], 1e-8, 1-1e-8))
)

diag_train["oof_pred_label_05"] = (diag_train["oof_pred_proba"] >= 0.5).astype(int)
diag_train["oof_error_05"] = (diag_train["oof_pred_label_05"] != diag_train["y"]).astype(int)

diag_train.head()

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, brier_score_loss

RAW_DIR = Path("../Data/Raw")
diag = pd.read_csv(RAW_DIR / "triage_train_diagnostics.csv")

print(diag.shape)
print(diag.columns.tolist())

p = diag["oof_pred_proba"]
y = diag["y"]
pred = (p >= 0.5).astype(int)

print("AUROC:", roc_auc_score(y, p))
print("AUPRC:", average_precision_score(y, p))
print("Recall:", recall_score(y, pred))
print("Precision:", precision_score(y, pred))
print("Brier:", brier_score_loss(y, p))

False negative and high-confidence contradiction flags

In [ ]:
diag_train["false_negative_05"] = (
    (diag_train["y"] == 1) & (diag_train["oof_pred_label_05"] == 0)
).astype(int)

diag_train["high_conf_contradiction"] = (
    ((diag_train["oof_pred_proba"] >= 0.80) & (diag_train["y"] == 0)) |
    ((diag_train["oof_pred_proba"] <= 0.20) & (diag_train["y"] == 1))
).astype(int)

print("FN rate:", diag_train["false_negative_05"].mean())
print("High-conf contradiction rate:", diag_train["high_conf_contradiction"].mean())

High-loss pool

In [ ]:
loss_cutoff = diag_train["oof_logloss"].quantile(0.90)
diag_train["high_loss_top10"] = (diag_train["oof_logloss"] >= loss_cutoff).astype(int)

print("Loss cutoff:", loss_cutoff)
print(diag_train["high_loss_top10"].mean())

Quick diagnostic plots

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(diag_train["oof_pred_proba"], bins=50)
plt.title("OOF Predicted Probability Distribution")
plt.show()

plt.figure(figsize=(8,4))
plt.hist(diag_train["oof_logloss"], bins=50)
plt.title("OOF Log-Loss Distribution")
plt.show()

Save diagnostic training table

In [ ]:
diag_train.to_csv("triage_train_diagnostics.csv", index=False)
print("Saved triage_train_diagnostics.csv")